# Historique des données XAUUSD

État du travail au **24 novembre 2026**.

Nous avons récupéré deux exports de ticks, extrait les CSV, mesuré leur couverture réelle avec DuckDB et comparé leur chevauchement. Les **43 478 752 ticks complets en commun sont identiques ligne par ligne**. Une fusion des deux sources donnerait **505 295 814 ticks complets**.

[Dossier Google Drive](https://drive.google.com/drive/folders/1a_HmvMG4HgrjlRCYAx0PYwyJrZNangCr) · [Dépôt Multi-Agents-Trading](https://github.com/hamzaelattar/Multi-Agents-Trading)

In [4]:
from pathlib import Path
import csv
import io
import os
import runpy
import subprocess
from tempfile import TemporaryDirectory

projet = Path(os.environ.get(
    "XAUUSD_PROJECT_DIR", str(Path.home() / "OneDrive/Desktop/Multi-Agents-Trading")
))
donnees = projet / "data_trading_agent"
ancien = donnees / "XAUUSD_Ticks_2004.01.01_2021.10.15.csv"
recent = donnees / "XAUUSD_Ticks_2021.01.04_2026.11.20.csv"
duckdb_exe = Path(os.environ.get(
    "DUCKDB_EXE", str(Path(os.environ.get("LOCALAPPDATA", ".")) / "duckdb/cli/2.0.1/duckdb.exe")
))
audit_complet = os.environ.get("XAUUSD_AUDIT_COMPLET", "0") == "1"


def litteral_sql(valeur):
    return "'" + str(valeur).replace("'", "''") + "'"


def lire_lignes(chemin, *, entete=True, parallele=False):
    return (
        f"read_csv({litteral_sql(chemin.as_posix())}, columns={{'ligne':'VARCHAR'}}, "
        f"delim='|', quote='', escape='', header={str(entete).lower()}, "
        f"auto_detect=false, parallel={str(parallele).lower()})"
    )


def interroger(sql):
    resultat = subprocess.run(
        [str(duckdb_exe), "-batch", "-bail", "-csv"], input=sql, text=True,
        encoding="utf-8", capture_output=True,
    )
    if resultat.returncode:
        raise RuntimeError(resultat.stderr.strip())
    return list(csv.DictReader(io.StringIO(resultat.stdout)))


def fragment_final(flux):
    fin = flux.seek(0, 2)
    flux.seek(max(0, fin - 4096))
    return flux.read().rsplit(b"\n", 1)[-1]


print(interroger("SELECT version() AS version;"))
for chemin in (ancien, recent):
    print(f"\n{chemin.name} · {chemin.stat().st_size:,} octets")
    exemple = interroger(f"SELECT ligne FROM {lire_lignes(chemin)} LIMIT 1;")
    print(exemple[0]["ligne"])

[{'version': 'v2.0.1'}]

XAUUSD_Ticks_2004.01.01_2021.10.15.csv · 25,145,311,232 octets
2004.01.01 02:00:02.937,415,332,414,922,0,00048,0,00026

XAUUSD_Ticks_2021.01.04_2026.11.20.csv · 6,830,074,333 octets
2021.01.04 01:00:00.413,1905,366,1904,998,0,00037,0,00015


## 2. Couverture et comptage

Dates et heures telles qu’étiquetées dans les exports (`EET`). Les totaux de ticks excluent l’en-tête et, dans le fichier ancien, la dernière ligne incomplète.

| Export | Premier tick | Dernier tick complet | Ticks complets | Lignes de données |
|---|---|---|---:|---:|
| Ancien | 01/01/2004 02:00:02.937 | 15/10/2021 23:54:59.188 | 432 697 435 | 432 697 436 |
| Récent | 04/01/2021 01:00:00.413 | 20/11/2026 06:00:02.344 | 116 077 131 | 116 077 131 |

Le premier CSV compte **432 697 437 lignes en incluant l’en-tête**. Il contient une ligne finale tronquée.

Les résultats ci-dessous ont été enregistrés après exécution locale. Pour relancer les lectures complètes des deux CSV et la comparaison, définir `audit_complet = True` dans la cellule de configuration, puis réexécuter le notebook. Le comptage lit environ 32 Go de CSV et la comparaison utilise un fichier temporaire d’environ 2,5 Go.

In [5]:
def compter(chemin):
    with chemin.open("rb") as flux:
        fragment = fragment_final(flux)
    complet = f"ligne IS DISTINCT FROM {litteral_sql(fragment.decode())}" if fragment else "true"
    return interroger(f"""
        SET threads = 3;
        SELECT count(*) AS lignes_de_donnees,
               count(*) FILTER (WHERE {complet}) AS ticks_complets,
               min(left(ligne, 23)) FILTER (WHERE {complet}) AS premier_tick,
               max(left(ligne, 23)) FILTER (WHERE {complet}) AS dernier_tick_complet
        FROM {lire_lignes(chemin, parallele=True)};
    """)[0]


if audit_complet:
    for chemin in (ancien, recent):
        print(chemin.name)
        for nom, valeur in compter(chemin).items():
            print(f"  {nom} = {valeur}")

## 3. Comparaison exacte du chevauchement

L’export récent commence le **4 janvier 2021 à 01:00:00.413**. Le code de fusion localise cet horodatage dans l’ancien export. DuckDB compare ensuite les lignes complètes de cette fin de fichier avec le début de l’export récent via `POSITIONAL JOIN`. La comparaison conserve l’ordre des lignes et toutes les valeurs brutes.

| Résultat vérifié | Valeur |
|---|---:|
| Ticks complets communs | 43 478 752 |
| Lignes différentes | 0 |
| Ticks complets anciens avant le chevauchement | 389 218 683 |
| Ticks récents supplémentaires | 72 598 379 |

Les lignes communes vont jusqu’au **15 octobre 2021 à 23:54:59.188**. Juste après, les 53 octets de la ligne incomplète de l’ancien fichier correspondent au début du tick complet du **15 octobre 2021 à 23:55:00.082** dans le fichier récent. Ce tick complet fait partie des ticks récents supplémentaires.

In [6]:
def comparer_recouvrement():
    fusion = runpy.run_path(str(projet / "github/fusionner_ticks.py"))
    with TemporaryDirectory(prefix="xauusd_duckdb_") as dossier:
        extrait = Path(dossier) / "recouvrement_ancien.txt"
        with ancien.open("rb") as a, recent.open("rb") as b:
            premier_ancien = fusion["lire_entete"](a)
            premier_recent = fusion["lire_entete"](b)
            debut = fusion["trouver_debut_recouvrement"](a, premier_ancien, b.readline()[:23])
            fragment = fragment_final(a)
            fin = a.tell() - len(fragment)
            taille = fin - debut
            a.seek(debut)
            lignes = 0
            with extrait.open("wb") as sortie:
                while a.tell() < fin:
                    bloc = a.read(min(16 * 1024**2, fin - a.tell()))
                    if not bloc:
                        raise EOFError(ancien)
                    sortie.write(bloc)
                    lignes += bloc.count(b"\n")
            b.seek(premier_recent + taille)
            suivant = b.readline().rstrip(b"\r\n")

        commun = interroger(f"""
            SET threads = 1;
            SELECT count(a.ligne) AS lignes_anciennes,
                   count(b.ligne) AS lignes_recentes,
                   count(*) FILTER (WHERE a.ligne = b.ligne) AS identiques,
                   count(*) FILTER (WHERE a.ligne IS DISTINCT FROM b.ligne) AS differentes,
                   min(left(a.ligne, 23)) AS premier_tick_commun,
                   max(left(a.ligne, 23)) AS dernier_tick_commun
            FROM {lire_lignes(extrait, entete=False)} a
            POSITIONAL JOIN (
                SELECT ligne FROM {lire_lignes(recent)} LIMIT {lignes}
            ) b;
        """)[0]
        suite = interroger(f"""
            SELECT starts_with({litteral_sql(suivant.decode())},
                               {litteral_sql(fragment.decode())}) AS fragment_confirme,
                   length({litteral_sql(fragment.decode())}) AS longueur_fragment,
                   left({litteral_sql(suivant.decode())}, 23) AS horodatage_tick_recupere;
        """)[0]
    return commun | suite


if audit_complet:
    for nom, valeur in comparer_recouvrement().items():
        print(f"{nom} = {valeur}")

## 4. Où en est la préparation des données ?

**La fusion donnerait 505 295 814 ticks complets.** Ce total correspond à **389 218 683 ticks anciens avant le chevauchement + 116 077 131 ticks de l’export récent**. La fusion a été testée sur un petit exemple et le chevauchement a été vérifié sur les fichiers réels. La génération du fichier fusionné complet reste à faire.

Le script de fusion produit un CSV compressé de cotations brutes. Pour alimenter les trois scripts d’entraînement, il reste à convertir les virgules décimales, choisir la durée des bougies et agréger les ticks en colonnes **Open, High, Low, Close**. Les scripts d’entraînement ont été essayés sur des bougies synthétiques. Les données historiques sont au stade des deux CSV extraits et contrôlés.